# 02 · Build the governed semantic model

Creates a **Direct Lake** semantic model over the lakehouse tables from notebook 01, then adds everything that makes it the single trusted source: relationships, the date table, sort orders, hidden keys, a calendar hierarchy, **44 agreed measures** (including the sentences that rewrite themselves for any filter), and **row-level and object-level security**.

Run it in the same workspace as the lakehouse. Running it again rebuilds the model from scratch (`OVERWRITE = True`), so make model changes here, or in Git, and not by hand in the service.

In [ ]:
%pip install -q semantic-link-labs

In [ ]:
MODEL = "Capital Bikeshare"          # semantic model name
LAKEHOUSE = "bike_lakehouse"         # the lakehouse notebook 01 wrote to
WORKSPACE = None                     # None = this notebook's workspace
OVERWRITE = True
TABLES = ['dim_date', 'dim_hour', 'dim_member_type', 'dim_station', 'fact_trips', 'fact_daily_rentals', 'fact_hourly_rentals', 'security_user_region']

RELATIONSHIPS = [('fact_trips', 'start_date', 'dim_date', 'date', True),
 ('fact_trips', 'start_hour', 'dim_hour', 'hour', True),
 ('fact_trips', 'member_type', 'dim_member_type', 'member_type', True),
 ('fact_trips', 'start_station_key', 'dim_station', 'station_key', True),
 ('fact_trips', 'end_station_key', 'dim_station', 'station_key', False),
 ('fact_daily_rentals', 'date', 'dim_date', 'date', True),
 ('fact_hourly_rentals', 'date', 'dim_date', 'date', True),
 ('fact_hourly_rentals', 'hour', 'dim_hour', 'hour', True)]
SORT_BY = [('dim_date', 'month_name', 'month'),
 ('dim_date', 'month_short', 'month'),
 ('dim_date', 'weekday_name', 'weekday_num'),
 ('dim_date', 'weekday_short', 'weekday_num'),
 ('dim_date', 'year_month', 'year_month_sort'),
 ('dim_date', 'season', 'season_sort'),
 ('dim_hour', 'hour_label', 'hour'),
 ('dim_hour', 'hour_range', 'hour'),
 ('dim_hour', 'day_part', 'day_part_sort'),
 ('dim_member_type', 'member_type', 'member_type_sort')]
HIDDEN_COLUMNS = [('fact_trips', 'trip_id'),
 ('fact_trips', 'start_date'),
 ('fact_trips', 'start_hour'),
 ('fact_trips', 'member_type'),
 ('fact_trips', 'start_station_key'),
 ('fact_trips', 'end_station_key'),
 ('fact_daily_rentals', 'date'),
 ('fact_hourly_rentals', 'date'),
 ('fact_hourly_rentals', 'hour'),
 ('dim_date', 'year_month_sort'),
 ('dim_date', 'season_sort'),
 ('dim_hour', 'day_part_sort'),
 ('dim_member_type', 'member_type_sort'),
 ('dim_station', 'region_id')]
DO_NOT_SUM = [('dim_date', 'date_key'),
 ('dim_date', 'year'),
 ('dim_date', 'month'),
 ('dim_date', 'day_of_month'),
 ('dim_date', 'weekday_num'),
 ('dim_date', 'weather_code'),
 ('dim_date', 'year_month_sort'),
 ('dim_date', 'season_sort'),
 ('dim_hour', 'hour'),
 ('dim_hour', 'day_part_sort'),
 ('dim_station', 'station_key'),
 ('dim_station', 'latitude'),
 ('dim_station', 'longitude'),
 ('fact_trips', 'trip_id'),
 ('fact_trips', 'start_station_key'),
 ('fact_trips', 'end_station_key'),
 ('fact_trips', 'start_hour'),
 ('fact_hourly_rentals', 'hour'),
 ('fact_hourly_rentals', 'weather_code'),
 ('dim_member_type', 'member_type_sort')]
DATA_CATEGORY = [('dim_station', 'latitude', 'Latitude'), ('dim_station', 'longitude', 'Longitude')]
HIERARCHIES = [('dim_date', 'Calendar', ['year', 'quarter', 'month_name', 'date'])]
MEASURES = [('fact_trips', 'Trips', 'Trips', 'COUNTROWS ( fact_trips )', '#,0', 'Number of trips in the trip records, counted at the start station.'),
 ('fact_trips',
  'Trips',
  'Trips Ended',
  'CALCULATE ( [Trips], USERELATIONSHIP ( fact_trips[end_station_key], dim_station[station_key] ) )',
  '#,0',
  'Trips counted at the end station. Use with station fields to see arrivals.'),
 ('fact_trips',
  'Trips',
  'Member Trips',
  'CALCULATE ( [Trips], KEEPFILTERS ( dim_member_type[member_type] = "Member" ) )',
  '#,0',
  'Trips by registered members.'),
 ('fact_trips',
  'Trips',
  'Casual Trips',
  'CALCULATE ( [Trips], KEEPFILTERS ( dim_member_type[member_type] = "Casual" ) )',
  '#,0',
  'Trips by casual riders.'),
 ('fact_trips', 'Trips', 'Member Share', 'DIVIDE ( [Member Trips], [Trips] )', '0.0%', 'Share of trips taken by registered members.'),
 ('fact_trips',
  'Trips',
  'Round Trips',
  'CALCULATE ( [Trips], KEEPFILTERS ( fact_trips[is_round_trip] = TRUE () ) )',
  '#,0',
  'Trips that ended where they started.'),
 ('fact_trips', 'Trips', 'Average Trip Minutes', 'AVERAGE ( fact_trips[duration_min] )', '0.0', 'Average trip length in minutes.'),
 ('fact_trips',
  'Trips',
  'Median Trip Minutes',
  'MEDIAN ( fact_trips[duration_min] )',
  '0.0',
  'Middle trip length in minutes; less affected by very long trips.'),
 ('fact_trips',
  'Trips',
  'Active Stations',
  'DISTINCTCOUNT ( fact_trips[start_station_key] )',
  '#,0',
  'Stations where at least one trip started.'),
 ('fact_trips',
  'Trips',
  'Trips per Day',
  'DIVIDE ( [Trips], DISTINCTCOUNT ( fact_trips[start_date] ) )',
  '#,0',
  'Average trips per day that had trips.'),
 ('fact_trips',
  'Trips\\Downtown',
  'Downtown Departures',
  'CALCULATE ( [Trips], KEEPFILTERS ( dim_station[is_downtown] = TRUE () ) )',
  '#,0',
  'Trips that started at a downtown station.'),
 ('fact_trips',
  'Trips\\Downtown',
  'Downtown Arrivals',
  'CALCULATE ( [Trips], USERELATIONSHIP ( fact_trips[end_station_key], dim_station[station_key] ), KEEPFILTERS ( dim_station[is_downtown] '
  '= TRUE () ) )',
  '#,0',
  'Trips that ended at a downtown station.'),
 ('fact_trips',
  'Trips\\Downtown',
  'Downtown Net Arrivals',
  '[Downtown Arrivals] - [Downtown Departures]',
  '#,0;-#,0',
  'Arrivals minus departures downtown. Positive in the morning rush, negative in the evening.'),
 ('fact_trips',
  'Data quality',
  'Trips at Unplaced Stations',
  'CALCULATE ( [Trips], KEEPFILTERS ( dim_station[is_mapped] = FALSE () ) )',
  '#,0',
  'Trips starting at a station with no known position.'),
 ('fact_daily_rentals', 'Rentals', 'Rentals', 'SUM ( fact_daily_rentals[total] )', '#,0', 'All rentals in the UCI daily totals.'),
 ('fact_daily_rentals', 'Rentals', 'Registered Rentals', 'SUM ( fact_daily_rentals[registered] )', '#,0', 'Rentals by registered members.'),
 ('fact_daily_rentals', 'Rentals', 'Casual Rentals', 'SUM ( fact_daily_rentals[casual] )', '#,0', 'Rentals by casual riders.'),
 ('fact_daily_rentals',
  'Rentals',
  'Registered Share',
  'DIVIDE ( [Registered Rentals], [Rentals] )',
  '0.0%',
  'Share of rentals by registered members.'),
 ('fact_daily_rentals', 'Rentals', 'Days', 'COUNTROWS ( fact_daily_rentals )', '#,0', 'Number of days in the current selection.'),
 ('fact_daily_rentals', 'Rentals', 'Avg Daily Rentals', 'DIVIDE ( [Rentals], [Days] )', '#,0', 'Average rentals per day.'),
 ('fact_daily_rentals',
  'Rentals\\Rider slicer',
  'Selected Rider Rentals',
  'SWITCH ( SELECTEDVALUE ( dim_member_type[member_type] ), "Member", [Registered Rentals], "Casual", [Casual Rentals], [Rentals] )',
  '#,0',
  'Rentals for the rider type picked on a dim_member_type slicer (all riders when none or several are picked).'),
 ('fact_daily_rentals',
  'Rentals\\Rider slicer',
  'Selected Rider Avg Daily',
  'DIVIDE ( [Selected Rider Rentals], [Days] )',
  '#,0',
  'Average rentals per day for the picked rider type.'),
 ('fact_daily_rentals',
  'Rentals\\Year on year',
  'Selected Rider Avg Daily LY',
  'CALCULATE ( [Selected Rider Avg Daily], SAMEPERIODLASTYEAR ( dim_date[date] ) )',
  '#,0',
  'The same measure on the same dates a year earlier.'),
 ('fact_daily_rentals',
  'Rentals\\Year on year',
  'Selected Rider YoY %',
  'VAR firstDay = MIN ( fact_daily_rentals[date] )\n'
  'VAR cur = [Selected Rider Avg Daily]\n'
  'VAR ly = [Selected Rider Avg Daily LY]\n'
  'RETURN\n'
  '    IF ( firstDay >= DATE ( 2012, 1, 1 ) && NOT ISBLANK ( ly ), DIVIDE ( cur - ly, ly ) )',
  '+0.0%;-0.0%;0.0%',
  'Change in rentals per day against the same dates in 2011. Only shown when the selection is within 2012, so it never compares against '
  'partial data.'),
 ('fact_daily_rentals',
  'Rentals\\Extremes',
  'Busiest Day',
  'VAR t = FILTER ( ADDCOLUMNS ( VALUES ( dim_date[date] ), "@r", [Selected Rider Rentals] ), NOT ISBLANK ( [@r] ) )\n'
  'RETURN MAXX ( TOPN ( 1, t, [@r], DESC ), dim_date[date] )',
  'd mmm yyyy',
  'Day with the most rentals in the selection.'),
 ('fact_daily_rentals',
  'Rentals\\Extremes',
  'Busiest Day Rentals',
  'VAR t = FILTER ( ADDCOLUMNS ( VALUES ( dim_date[date] ), "@r", [Selected Rider Rentals] ), NOT ISBLANK ( [@r] ) )\n'
  'RETURN MAXX ( t, [@r] )',
  '#,0',
  'Rentals on the busiest day.'),
 ('fact_daily_rentals',
  'Rentals\\Extremes',
  'Quietest Day',
  'VAR t = FILTER ( ADDCOLUMNS ( VALUES ( dim_date[date] ), "@r", [Selected Rider Rentals] ), NOT ISBLANK ( [@r] ) )\n'
  'RETURN MAXX ( TOPN ( 1, t, [@r], ASC ), dim_date[date] )',
  'd mmm yyyy',
  'Day with the fewest rentals in the selection.'),
 ('fact_daily_rentals',
  'Rentals\\Extremes',
  'Quietest Day Rentals',
  'VAR t = FILTER ( ADDCOLUMNS ( VALUES ( dim_date[date] ), "@r", [Selected Rider Rentals] ), NOT ISBLANK ( [@r] ) )\n'
  'RETURN MINX ( t, [@r] )',
  '#,0',
  'Rentals on the quietest day.'),
 ('fact_daily_rentals',
  'Rentals\\Weather',
  'Avg Daily Rentals Clear',
  'CALCULATE ( [Selected Rider Avg Daily], KEEPFILTERS ( dim_date[weather_group] = "Clear" ) )',
  '#,0',
  'Average rentals per day on clear days.'),
 ('fact_daily_rentals',
  'Rentals\\Weather',
  'Avg Daily Rentals Rain or Snow',
  'CALCULATE ( [Selected Rider Avg Daily], KEEPFILTERS ( dim_date[weather_group] = "Rain or snow" ) )',
  '#,0',
  'Average rentals per day on days with rain or snow.'),
 ('fact_daily_rentals',
  'Rentals\\Weather',
  'Rain or Snow Drop %',
  'DIVIDE ( [Avg Daily Rentals Clear] - [Avg Daily Rentals Rain or Snow], [Avg Daily Rentals Clear] )',
  '0.0%',
  'How much lower rentals are on rain or snow days than on clear days.'),
 ('fact_hourly_rentals',
  'Hourly',
  'Hourly Rentals',
  'SWITCH ( SELECTEDVALUE ( dim_member_type[member_type] ), "Member", SUM ( fact_hourly_rentals[registered] ), "Casual", SUM ( '
  'fact_hourly_rentals[casual] ), SUM ( fact_hourly_rentals[total] ) )',
  '#,0',
  'Rentals in the hourly totals, for the picked rider type.'),
 ('fact_hourly_rentals',
  'Hourly',
  'Avg Rentals per Hour',
  'DIVIDE ( [Hourly Rentals], CALCULATE ( DISTINCTCOUNT ( fact_hourly_rentals[date] ), REMOVEFILTERS ( dim_hour ) ) )',
  '#,0',
  'Average rentals in each hour across the days selected. Put dim_hour on the axis.'),
 ('fact_hourly_rentals',
  'Hourly',
  'Morning Peak Hour',
  'VAR t = ADDCOLUMNS ( FILTER ( ALL ( dim_hour[hour] ), dim_hour[hour] >= 6 && dim_hour[hour] <= 9 ), "@r", [Avg Rentals per Hour] )\n'
  'VAR h = MAXX ( TOPN ( 1, t, [@r], DESC ), dim_hour[hour] )\n'
  'RETURN CALCULATE ( MAX ( dim_hour[hour_label] ), REMOVEFILTERS ( dim_hour ), dim_hour[hour] = h )',
  None,
  'Busiest hour between 6am and 10am.'),
 ('fact_hourly_rentals',
  'Hourly',
  'Morning Peak Rentals',
  'VAR t = ADDCOLUMNS ( FILTER ( ALL ( dim_hour[hour] ), dim_hour[hour] >= 6 && dim_hour[hour] <= 9 ), "@r", [Avg Rentals per Hour] )\n'
  'RETURN MAXX ( t, [@r] )',
  '#,0',
  'Average rentals in the morning peak hour.'),
 ('fact_hourly_rentals',
  'Hourly',
  'Evening Peak Hour',
  'VAR t = ADDCOLUMNS ( FILTER ( ALL ( dim_hour[hour] ), dim_hour[hour] >= 15 && dim_hour[hour] <= 19 ), "@r", [Avg Rentals per Hour] )\n'
  'VAR h = MAXX ( TOPN ( 1, t, [@r], DESC ), dim_hour[hour] )\n'
  'RETURN CALCULATE ( MAX ( dim_hour[hour_label] ), REMOVEFILTERS ( dim_hour ), dim_hour[hour] = h )',
  None,
  'Busiest hour between 3pm and 8pm.'),
 ('fact_hourly_rentals',
  'Hourly',
  'Evening Peak Rentals',
  'VAR t = ADDCOLUMNS ( FILTER ( ALL ( dim_hour[hour] ), dim_hour[hour] >= 15 && dim_hour[hour] <= 19 ), "@r", [Avg Rentals per Hour] )\n'
  'RETURN MAXX ( t, [@r] )',
  '#,0',
  'Average rentals in the evening peak hour.'),
 ('fact_hourly_rentals',
  'Hourly',
  'Peak Hour',
  'VAR t = ADDCOLUMNS ( ALL ( dim_hour[hour] ), "@r", [Avg Rentals per Hour] )\n'
  'VAR h = MAXX ( TOPN ( 1, t, [@r], DESC ), dim_hour[hour] )\n'
  'RETURN CALCULATE ( MAX ( dim_hour[hour_label] ), REMOVEFILTERS ( dim_hour ), dim_hour[hour] = h )',
  None,
  'Busiest hour of the day.'),
 ('fact_hourly_rentals',
  'Hourly',
  'Peak Hour Rentals',
  'VAR t = ADDCOLUMNS ( ALL ( dim_hour[hour] ), "@r", [Avg Rentals per Hour] )\nRETURN MAXX ( t, [@r] )',
  '#,0',
  'Average rentals in the busiest hour.'),
 ('fact_hourly_rentals',
  'Hourly',
  'Midday Average Rentals',
  'AVERAGEX ( FILTER ( ALL ( dim_hour[hour] ), dim_hour[hour] >= 10 && dim_hour[hour] <= 14 ), [Avg Rentals per Hour] )',
  '#,0',
  'Average rentals per hour from 10am to 3pm, used to tell a commuting pattern from a leisure one.'),
 ('fact_daily_rentals',
  'Narrative',
  'Summary Text',
  'VAR n = [Days]\n'
  'VAR d0 = MIN ( fact_daily_rentals[date] )\n'
  'VAR d1 = MAX ( fact_daily_rentals[date] )\n'
  'VAR who = SWITCH ( SELECTEDVALUE ( dim_member_type[member_type] ), "Member", "registered members", "Casual", "casual riders", "riders" '
  ')\n'
  'VAR yoy = [Selected Rider YoY %]\n'
  'VAR main =\n'
  '    IF ( n = 1,\n'
  '        "On " & FORMAT ( d0, "dddd d mmmm yyyy" ) & ", " & who & " took " & FORMAT ( [Selected Rider Rentals], "#,0" ) & " trips.",\n'
  '        "From " & FORMAT ( d0, "d mmm yyyy" ) & " to " & FORMAT ( d1, "d mmm yyyy" ) & ", " & who & " took " & FORMAT ( [Selected Rider '
  'Rentals], "#,0" )\n'
  '            & " trips, an average of " & FORMAT ( [Selected Rider Avg Daily], "#,0" ) & " a day across " & FORMAT ( n, "#,0" ) & " '
  'days." )\n'
  'VAR share = IF ( ISBLANK ( SELECTEDVALUE ( dim_member_type[member_type] ) ), " Registered members made " & FORMAT ( [Registered Share], '
  '"0.0%" ) & " of them." )\n'
  'VAR change = IF ( NOT ISBLANK ( yoy ), " Per day, that is " & IF ( yoy >= 0, "up ", "down " ) & FORMAT ( ABS ( yoy ), "0.0%" ) & " on '
  'the same dates in 2011." )\n'
  'RETURN IF ( ISBLANK ( n ), "No days match these filters.", main & share & change )',
  None,
  'Opening sentence of the report. It rewrites itself for whatever is selected.'),
 ('fact_daily_rentals',
  'Narrative',
  'Extremes Text',
  'IF ( [Days] > 1,\n'
  '    "The busiest day was " & FORMAT ( [Busiest Day], "dddd d mmmm yyyy" ) & ", with " & FORMAT ( [Busiest Day Rentals], "#,0" ) & " '
  'trips. The quietest was "\n'
  '        & FORMAT ( [Quietest Day], "dddd d mmmm yyyy" ) & ", with " & FORMAT ( [Quietest Day Rentals], "#,0" )\n'
  '        & IF ( [Quietest Day] = DATE ( 2012, 10, 29 ), ", the day Hurricane Sandy reached Washington.", "." ) )',
  None,
  'The busiest and quietest days in the selection.'),
 ('fact_hourly_rentals',
  'Narrative',
  'Rush Hour Text',
  'VAR am = [Morning Peak Rentals]\n'
  'VAR pm = [Evening Peak Rentals]\n'
  'VAR mid = [Midday Average Rentals]\n'
  'RETURN\n'
  '    SWITCH ( TRUE (),\n'
  '        ISBLANK ( [Peak Hour Rentals] ) || [Peak Hour Rentals] < 3, "There are too few trips in this selection to show a clear daily '
  'rhythm.",\n'
  '        am > mid * 1.4 && pm > mid * 1.2,\n'
  '            "Riding has two rush hours: a morning peak at " & [Morning Peak Hour] & " with " & FORMAT ( am, "#,0" ) & " trips in the '
  'hour, and an evening peak at "\n'
  '                & [Evening Peak Hour] & " with " & FORMAT ( pm, "#,0" ) & ". That is the shape of commuting.",\n'
  '        "There is no real rush hour here. Riding peaks at " & [Peak Hour] & ", with " & FORMAT ( [Peak Hour Rentals], "#,0" ) & " trips '
  'in the hour. That is the shape of leisure riding." )',
  None,
  'Describes the daily rhythm: commuting (two peaks) or leisure (one afternoon peak).'),
 ('fact_daily_rentals',
  'Narrative',
  'Weather Text',
  'VAR c = [Avg Daily Rentals Clear]\n'
  'VAR w = [Avg Daily Rentals Rain or Snow]\n'
  'RETURN\n'
  '    IF ( NOT ISBLANK ( c ) && NOT ISBLANK ( w ),\n'
  '        "Weather makes a big difference. Clear days averaged " & FORMAT ( c, "#,0" ) & " trips, against " & FORMAT ( w, "#,0" )\n'
  '            & " on days with rain or snow, a drop of " & FORMAT ( [Rain or Snow Drop %], "0.0%" ) & "." )',
  None,
  'Compares clear days with rain or snow days, when the selection has both.')]
ROLES = [('All data', 'Read', 'Analysts and owners: no row filter.', [], []),
 ('Region viewers',
  'Read',
  'Sees trips only for stations in the regions listed against their sign-in in security_user_region. Bike numbers are hidden. City-wide '
  'UCI daily and hourly totals have no station, so they stay visible in full.',
  [('dim_station',
    '[region_name] IN CALCULATETABLE ( VALUES ( security_user_region[region_name] ), security_user_region[user_email] = LOWER ( '
    'USERPRINCIPALNAME () ) )'),
   ('security_user_region', '[user_email] = LOWER ( USERPRINCIPALNAME () )')],
  [('fact_trips', 'bike_number', 'None')])]

In [ ]:
import sempy.fabric as fabric
from sempy_labs.directlake import generate_direct_lake_semantic_model

generate_direct_lake_semantic_model(dataset=MODEL, tables=TABLES, source=LAKEHOUSE, source_type="Lakehouse",
                                    workspace=WORKSPACE, refresh=False, overwrite=OVERWRITE)
print("Created", MODEL)

In [ ]:
from sempy_labs.tom import connect_semantic_model

with connect_semantic_model(dataset=MODEL, readonly=False, workspace=WORKSPACE) as tom:
    T = tom.model.Tables

    have = {(r.FromTable.Name, r.FromColumn.Name, r.ToTable.Name, r.ToColumn.Name) for r in tom.model.Relationships}
    for ft, fc, tt, tc, active in RELATIONSHIPS:
        if (ft, fc, tt, tc) not in have:
            tom.add_relationship(from_table=ft, from_column=fc, to_table=tt, to_column=tc,
                                 from_cardinality="Many", to_cardinality="One", is_active=active)

    tom.mark_as_date_table(table_name="dim_date", column_name="date")

    for t, c, s in SORT_BY:
        T[t].Columns[c].SortByColumn = T[t].Columns[s]
    for t, c in HIDDEN_COLUMNS:
        T[t].Columns[c].IsHidden = True
    for t, c in DO_NOT_SUM:
        tom.set_summarize_by(table_name=t, column_name=c, value="None")
    for t, c, cat in DATA_CATEGORY:
        T[t].Columns[c].DataCategory = cat
    T["security_user_region"].IsHidden = True

    for t, name, cols in HIERARCHIES:
        if name not in [h.Name for h in T[t].Hierarchies]:
            tom.add_hierarchy(table_name=t, hierarchy_name=name, columns=cols)

    have_m = {m.Name for m in tom.all_measures()}
    for home, folder, name, expr, fmt, desc in MEASURES:
        if name in have_m:
            m = T[home].Measures[name]
            m.Expression, m.DisplayFolder, m.Description = expr, folder, desc
            if fmt: m.FormatString = fmt
        else:
            tom.add_measure(table_name=home, measure_name=name, expression=expr, format_string=fmt,
                            display_folder=folder, description=desc)

    have_r = {r.Name for r in tom.model.Roles}
    for role, permission, desc, filters, ols in ROLES:
        if role not in have_r:
            tom.add_role(role_name=role, model_permission=permission, description=desc)
        for t, f in filters:
            tom.set_rls(role_name=role, table_name=t, filter_expression=f)
        for t, c, p in ols:
            tom.set_ols(role_name=role, table_name=t, column_name=c, permission=p)

print("Model updated: relationships, date table, formatting, measures and security.")

In [ ]:
fabric.refresh_dataset(dataset=MODEL, workspace=WORKSPACE)
print("Refresh requested. Give it a minute, then run the checks below.")

In [ ]:
# Every number here should match the dashboard and the expected values.
q = '''
EVALUATE
ROW (
    "Trips", [Trips], "Rentals", [Rentals], "Member Share", [Member Share], "Registered Share", [Registered Share],
    "Busiest Day", [Busiest Day], "Busiest Day Rentals", [Busiest Day Rentals],
    "Quietest Day", [Quietest Day], "Quietest Day Rentals", [Quietest Day Rentals],
    "Morning Peak Hour", [Morning Peak Hour], "Evening Peak Hour", [Evening Peak Hour],
    "Active Stations", [Active Stations], "Summary Text", [Summary Text]
)'''
display(fabric.evaluate_dax(dataset=MODEL, dax_string=q, workspace=WORKSPACE))
print('''Expected: Trips 3,255,678 · Rentals 3,292,679 · Member Share 81.0% · Registered Share 81.2%
Busiest 15 Sep 2012 (8,714) · Quietest 29 Oct 2012 (22) · Morning peak 8am · Evening peak 5pm · Active stations 194''')

q2 = '''
EVALUATE
TOPN ( 3, SUMMARIZECOLUMNS ( dim_station[station_name], "Trips", [Trips] ), [Trips], DESC )'''
display(fabric.evaluate_dax(dataset=MODEL, dax_string=q2, workspace=WORKSPACE))
print("Expected top station: Massachusetts Ave & Dupont Circle NW, 107,162")

q3 = '''
EVALUATE
CALCULATETABLE (
    ROW ( "Downtown arrivals 8am", [Downtown Arrivals], "Downtown departures 8am", [Downtown Departures] ),
    dim_hour[hour] = 8, dim_date[is_working_day] = TRUE ()
)'''
display(fabric.evaluate_dax(dataset=MODEL, dax_string=q3, workspace=WORKSPACE))
print("Expected over the 500 working days: 91,405 arrivals and 32,618 departures")

**Next:** follow the governance guide: give people access through workspace roles, add members to the two security roles in the model's *Security* settings, apply a sensitivity label, endorse the model, and connect the workspace to Git.